# 实验三：单卡训练代码结构解析

本章阅读 `src/scripts/train_yolo_single_npu_amp.py` 入口及其复用的训练实现。虽然底层脚本保留了 DDP 扩展能力，但当前硬件只有 1 张 NPU，所以实际运行路径是单进程、单卡、`world_size=1`。

训练脚本并不是为了追求最复杂的 YOLO 实现，而是为了让你看清楚大规模训练调优中的固定骨架：数据读取、模型前向、loss、AMP、warmup、日志、checkpoint、profiling。


## 训练主流程

<table style="margin-left: 0; margin-right: auto; text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">阶段</th>
      <th style="text-align: left;">代码位置</th>
      <th style="text-align: left;">作用</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">读取配置</td>
      <td style="text-align: left;"><code>load_config</code></td>
      <td style="text-align: left;">读取 YAML 中的数据路径、类别数和训练超参数</td>
    </tr>
    <tr>
      <td style="text-align: left;">选择设备</td>
      <td style="text-align: left;"><code>select_device</code></td>
      <td style="text-align: left;">优先选择 Ascend NPU，其次 CUDA，最后 CPU</td>
    </tr>
    <tr>
      <td style="text-align: left;">构建 Dataset</td>
      <td style="text-align: left;"><code>make_dataset</code></td>
      <td style="text-align: left;">从 VOC split 列表读取图片，并读取 YOLO txt label</td>
    </tr>
    <tr>
      <td style="text-align: left;">构建 DataLoader</td>
      <td style="text-align: left;"><code>DataLoader(...)</code></td>
      <td style="text-align: left;">并行读图、增强、组 batch</td>
    </tr>
    <tr>
      <td style="text-align: left;">构建模型</td>
      <td style="text-align: left;"><code>TinyYolo(...)</code></td>
      <td style="text-align: left;">教学用 YOLO 风格检测网络</td>
    </tr>
    <tr>
      <td style="text-align: left;">AMP</td>
      <td style="text-align: left;"><code>make_amp(...)</code></td>
      <td style="text-align: left;">混合精度上下文与梯度缩放</td>
    </tr>
    <tr>
      <td style="text-align: left;">Warmup</td>
      <td style="text-align: left;"><code>scheduled_lr(...)</code></td>
      <td style="text-align: left;">前期逐步升高学习率，减少发散风险</td>
    </tr>
    <tr>
      <td style="text-align: left;">保存模型</td>
      <td style="text-align: left;"><code>save_checkpoint(...)</code></td>
      <td style="text-align: left;">输出 <code>last.pt</code> 和周期性 checkpoint</td>
    </tr>
  </tbody>
</table>


In [ ]:
# ====== 1. 查看训练入口文件 ======
from pathlib import Path

entry = Path('src/scripts/train_yolo_single_npu_amp.py')
print(entry.read_text(encoding='utf-8'))


## VOC split 如何进入训练

配置文件中最关键的数据字段是：

```yaml
data:
  root: /mnt/workspace/datasets/voc
  train_images: images
  train_labels: labels
  train_list: splits/train.txt
  val_images: images
  val_labels: labels
  val_list: splits/val.txt
```

训练脚本优先读取 `train_list`。这样做的好处是：不用移动或复制大量图片，只要在 split 文件中列出相对路径，例如 `images/train2007/000005.jpg`。脚本会自动寻找对应的 `labels/train2007/000005.txt`。


In [ ]:
# ====== 2. 查看 Dataset 相关代码片段 ======
from pathlib import Path

script = Path('src/scripts/train_yolo_ddp_amp.py')
text = script.read_text(encoding='utf-8')
start = text.index('class YoloTxtDataset')
end = text.index('class SyntheticYoloDataset')
print(text[start:end])


## 单卡训练的数据流

```text
VOC split 列表
  -> YoloTxtDataset 读取图片与 YOLO txt
  -> DataLoader 按 batch 组装
  -> 图片张量移动到 npu:0
  -> 模型前向计算
  -> YOLO loss
  -> AMP 反向传播
  -> optimizer.step()
  -> 日志和 checkpoint
```

没有多卡时，不会触发 HCCL AllReduce，也不会计算扩展效率。此时最值得观察的是单卡吞吐、DataLoader 是否供得上数据、AMP 是否稳定，以及 Host 到 NPU 之间是否存在明显等待。


In [ ]:
# ====== 3. 查看 warmup 学习率调度 ======
from pathlib import Path

text = Path('src/scripts/train_yolo_ddp_amp.py').read_text(encoding='utf-8')
start = text.index('def scheduled_lr')
end = text.index('def set_lr')
print(text[start:end])


## 哪些代码是为 DDP 扩展预留的

<table style="margin-left: 0; margin-right: auto; text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">代码</th>
      <th style="text-align: left;">单卡时状态</th>
      <th style="text-align: left;">多卡时作用</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><code>init_process_group</code></td>
      <td style="text-align: left;">不执行</td>
      <td style="text-align: left;">建立 HCCL/NCCL/Gloo 进程组</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>DistributedSampler</code></td>
      <td style="text-align: left;">不启用</td>
      <td style="text-align: left;">每个 rank 读取不同数据切片</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>DistributedDataParallel</code></td>
      <td style="text-align: left;">不封装</td>
      <td style="text-align: left;">backward 后自动同步梯度</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>rank0()</code></td>
      <td style="text-align: left;">始终为真</td>
      <td style="text-align: left;">多卡时只让 rank 0 打印日志和保存模型</td>
    </tr>
  </tbody>
</table>

保留这些代码，是为了以后从单卡迁移到多卡时不用重写训练脚本。当前实验先把单卡训练和调优指标做扎实。


## 课后练习

请根据本节实验内容完成以下练习。题型包含单选题、多选题、判断题、填空题、简答题和代码设计题。

1. (单选题) `YoloTxtDataset` 最主要的职责是？
   - A. 从 VOC/YOLO 目录读取图片和 txt 标注并组成训练样本
   - B. 调用 ATC 转 OM
   - C. 执行 Git push
   - D. 生成 MindStudio 报告

2. (单选题) `collate_yolo` 的作用更接近哪一项？
   - A. 把一个 batch 中不同数量的目标框整理成可训练结构
   - B. 压缩 checkpoint
   - C. 修改学习率配置文件
   - D. 创建 Pull Request

3. (单选题) `TinyYolo` 在本实验中的定位是？
   - A. 教学用轻量检测模型
   - B. 完整 YOLOv5 官方大模型
   - C. 数据下载器
   - D. Git LFS 客户端

4. (单选题) `validate` 函数通常用于？
   - A. 训练后或训练间隔评估模型输出/损失
   - B. 转换 XML 文件编码
   - C. 设置 Git 用户名
   - D. 安装 CANN

5. (多选题) 训练主流程通常包含哪些步骤？
   - A. 读取配置
   - B. 构建 Dataset/DataLoader
   - C. 创建模型和优化器
   - D. 训练、验证并保存 checkpoint

6. (多选题) 代码中为 DDP 扩展预留的常见设计包括哪些？
   - A. rank/local_rank 参数
   - B. DistributedSampler
   - C. rank0 日志与保存
   - D. HCCL 后端初始化入口

7. (多选题) 目标检测训练中，一个样本通常至少需要哪些信息？
   - A. 图像张量
   - B. 目标类别
   - C. 目标框坐标
   - D. 训练 split 归属

8. (判断题) 训练代码结构清晰的好处之一，是后续定位 loss 异常或数据加载瓶颈更容易。

9. (判断题) 只要模型能 forward，就不需要验证 DataLoader 输出的 shape 和 dtype。

10. (填空题) PyTorch 中负责按 batch 迭代数据的常用组件是 `____`。

11. (填空题) 为了避免多进程重复保存 checkpoint，通常只在 `rank == ____` 时写文件。

12. (简答题) 为什么目标检测任务需要自定义 `collate_fn`？

13. (简答题) 训练脚本中配置文件相比硬编码参数有什么优势？

14. (简答题) 如果 loss 一直不变，应该从代码结构的哪些模块开始排查？

15. (代码设计题) 写一段伪代码，说明训练循环中 forward、loss、backward、step 的基本顺序。

> 参考答案见 answer/03.04_training_code_structure_answer.ipynb。
